In [ ]:
# =============================================================================
# Scaffold-grouped 10-fold CV: fold ROC/PR curves + mean curve (±1 SD band)
#   - Works with your existing ChEMBL CSV and RDKit Morgan FP pipeline
#   - Saves publication-ready PDF + PNG
#
# Requirements (openms_env):
#   pip install numpy pandas scikit-learn matplotlib rdkit-pypi umap-learn
# (UMAP not needed here)
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import AllChem, DataStructs


# --------------------------- Paths / I/O -------------------------------------

BASE = r"C:\Users\Besitzer\Desktop\M3_databases"
TRAIN_CSV = os.path.join(BASE, "ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv")
FIGDIR = os.path.join(BASE, "figures", "CV_curves")
os.makedirs(FIGDIR, exist_ok=True)

OUT_ROC_PREFIX = os.path.join(FIGDIR, "CV10_scaffold_ROC")
OUT_PR_PREFIX  = os.path.join(FIGDIR, "CV10_scaffold_PR")


# --------------------------- Helpers -----------------------------------------

def smiles_to_mol(smiles: str):
    if pd.isna(smiles):
        return None
    m = Chem.MolFromSmiles(str(smiles))
    return m

def murcko_scaffold_smiles(mol):
    if mol is None:
        return None
    scaf = MurckoScaffold.GetScaffoldForMol(mol)
    if scaf is None:
        return None
    return Chem.MolToSmiles(scaf)

def morgan_fp(mol, radius=2, n_bits=2048, use_chirality=True, use_features=False):
    # RDKit bit vector
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(
        mol, radius, nBits=n_bits, useChirality=use_chirality, useFeatures=use_features
    )

def fps_to_numpy(bitvect_list):
    """Convert RDKit ExplicitBitVect list -> numpy uint8 (n, n_bits)."""
    n = len(bitvect_list)
    n_bits = bitvect_list[0].GetNumBits()
    X = np.zeros((n, n_bits), dtype=np.uint8)
    for i, bv in enumerate(bitvect_list):
        arr = np.zeros((n_bits,), dtype=np.int8)
        DataStructs.ConvertToNumpyArray(bv, arr)
        X[i, :] = arr.astype(np.uint8)
    return X

def _save_pub(fig, out_prefix, dpi=300):
    png = out_prefix + ".png"
    pdf = out_prefix + ".pdf"
    fig.savefig(png, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"[SAVED] {png}")    
    print(f"[SAVED] {pdf}")

def _style_axes(ax):
    # Remove top/right spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Left & bottom spines black
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")

    # Tick color + tick label color
    ax.tick_params(axis="both", which="major",
                   labelsize=11,
                   colors="black")

    # Axis label colors
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")

    # Title color
    ax.title.set_color("black")


# --------------------------- Load + build features ---------------------------

df = pd.read_csv(TRAIN_CSV)
print("[LOAD]", TRAIN_CSV, "| shape =", df.shape)

# Expect at least: smiles, consensus_label
if "smiles" not in df.columns:
    raise ValueError("Expected a 'smiles' column in training CSV.")
if "consensus_label" not in df.columns:
    raise ValueError("Expected a 'consensus_label' column in training CSV.")

# Binary label
POS = {"active", "active_single"}
NEG = {"inactive", "inactive_single"}

df = df[df["consensus_label"].isin(POS | NEG)].copy()
df["y"] = df["consensus_label"].isin(POS).astype(int)

# Sample weights (your scheme)
w_map = {"active": 1.0, "inactive": 1.0, "active_single": 0.5, "inactive_single": 0.7}
df["w"] = df["consensus_label"].map(w_map).astype(float)

# RDKit mol, scaffold
print("[RDKit] building mols + scaffolds ...")
df["mol"] = df["smiles"].map(smiles_to_mol)
df = df[df["mol"].notna()].copy()

df["scaffold"] = df["mol"].map(murcko_scaffold_smiles)
df = df[df["scaffold"].notna()].copy()

# Fingerprints
print("[RDKit] building fingerprints ...")
fps = [morgan_fp(m, radius=2, n_bits=2048, use_chirality=True, use_features=False) for m in df["mol"]]
mask_ok = [fp is not None for fp in fps]
df = df.loc[mask_ok].copy()
fps = [fp for fp in fps if fp is not None]

X = fps_to_numpy(fps).astype(np.float32)  # float for sklearn
y = df["y"].values.astype(int)
groups = df["scaffold"].values
w = df["w"].values.astype(float)

print("[DATA] n =", len(df), "| pos =", int(y.sum()), "| neg =", int((y == 0).sum()),
      "| pos_frac =", float(y.mean()))
print("[SCAFF] unique scaffolds =", pd.Series(groups).nunique())


# --------------------------- Model pipeline ----------------------------------

# FP-only (your baseline)
pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),  # sparse-friendly
    ("clf", LogisticRegression(
        solver="liblinear",
        max_iter=5000,
        class_weight="balanced",
        penalty="l2",
        C=1.0,
    ))
])


# --------------------------- CV curves ---------------------------------------

def cv10_mean_roc_pr_curves(pipe, X, y, groups, sample_weight=None, n_splits=10):
    """
    Returns dict with:
      - roc: fold_fpr, fold_tpr, fold_auc, mean_fpr, mean_tpr, std_tpr, mean_auc, std_auc
      - pr:  fold_rec, fold_prec, fold_ap, mean_rec, mean_prec, std_prec, mean_ap, std_ap, baseline_prec
    Notes:
      - Mean ROC uses interpolation of TPR over common FPR grid.
      - Mean PR uses interpolation of precision over common recall grid.
    """
    gkf = GroupKFold(n_splits=n_splits)
    splits = list(gkf.split(X, y, groups=groups))

    # Common grids
    mean_fpr = np.linspace(0.0, 1.0, 401)
    mean_rec = np.linspace(0.0, 1.0, 401)

    fold_fprs, fold_tprs, fold_aucs = [], [], []
    fold_recs, fold_precs, fold_aps = [], [], []

    # Baseline precision (global prevalence)
    baseline_prec = float(y.mean())

    for k, (tr, te) in enumerate(splits, start=1):
        X_tr, y_tr = X[tr], y[tr]
        X_te, y_te = X[te], y[te]

        if sample_weight is not None:
            w_tr = sample_weight[tr]
            pipe.fit(X_tr, y_tr, clf__sample_weight=w_tr)
        else:
            pipe.fit(X_tr, y_tr)

        # probability for positive class
        p_te = pipe.predict_proba(X_te)[:, 1]

        # ROC
        fpr, tpr, _ = roc_curve(y_te, p_te)
        roc_auc = auc(fpr, tpr)
        fold_fprs.append(fpr)
        fold_tprs.append(tpr)
        fold_aucs.append(roc_auc)

        # PR
        prec, rec, _ = precision_recall_curve(y_te, p_te)
        ap = average_precision_score(y_te, p_te)
        fold_recs.append(rec)
        fold_precs.append(prec)
        fold_aps.append(ap)

        print(f"[Fold {k:02d}] ROC-AUC={roc_auc:.3f} | AP(PR-AUC)={ap:.3f} | n_test={len(te)} | pos_test={int(y_te.sum())}")

    # Aggregate ROC (TPR over FPR grid)
    tpr_mat = np.vstack([np.interp(mean_fpr, fold_fprs[i], fold_tprs[i]) for i in range(len(splits))])
    mean_tpr = np.mean(tpr_mat, axis=0)
    std_tpr = np.std(tpr_mat, axis=0)
    mean_tpr[0] = 0.0
    mean_tpr[-1] = 1.0

    mean_auc = float(np.mean(fold_aucs))
    std_auc = float(np.std(fold_aucs))

    # Aggregate PR (precision over recall grid)
    prec_mat = np.vstack([np.interp(mean_rec, fold_recs[i], fold_precs[i]) for i in range(len(splits))])
    mean_prec = np.mean(prec_mat, axis=0)
    std_prec = np.std(prec_mat, axis=0)

    mean_ap = float(np.mean(fold_aps))
    std_ap = float(np.std(fold_aps))

    return {
        "roc": dict(
            fold_fpr=fold_fprs, fold_tpr=fold_tprs, fold_auc=fold_aucs,
            mean_fpr=mean_fpr, mean_tpr=mean_tpr, std_tpr=std_tpr,
            mean_auc=mean_auc, std_auc=std_auc
        ),
        "pr": dict(
            fold_rec=fold_recs, fold_prec=fold_precs, fold_ap=fold_aps,
            mean_rec=mean_rec, mean_prec=mean_prec, std_prec=std_prec,
            mean_ap=mean_ap, std_ap=std_ap,
            baseline_prec=baseline_prec
        )
    }

res = cv10_mean_roc_pr_curves(pipe, X, y, groups, sample_weight=w, n_splits=10)


# --------------------------- Plot ROC ----------------------------------------
def plot_cv_roc(res_roc, out_prefix, title="Scaffold-grouped 10-fold CV ROC"):
    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    # Fold curves (light grey)
    for fpr, tpr, fold_auc in zip(res_roc["fold_fpr"], res_roc["fold_tpr"], res_roc["fold_auc"]):
        ax.plot(fpr, tpr, linewidth=1, alpha=0.25, color="grey")

    # Mean curve (blue)
    mfpr = res_roc["mean_fpr"]
    mtpr = res_roc["mean_tpr"]
    stpr = res_roc["std_tpr"]

    ax.plot(
        mfpr, mtpr,
        linewidth=2.5,
        color="blue",
        label=f"Mean ROC (AUC={res_roc['mean_auc']:.3f}±{res_roc['std_auc']:.3f})"
    )

    # SD band (light blue)
    ax.fill_between(
        mfpr,
        np.clip(mtpr - stpr, 0, 1),
        np.clip(mtpr + stpr, 0, 1),
        color="lightblue",
        alpha=0.30,
        linewidth=0
    )

    # Chance line (RED dashed)
    ax.plot(
        [0, 1], [0, 1],
        linestyle="--",
        linewidth=1.5,
        color="red",
        label="Chance"
    )

    ax.set_xlabel("False Positive Rate", fontsize=12, color="black")
    ax.set_ylabel("True Positive Rate", fontsize=12, color="black")
    ax.set_title(title, fontsize=13, color="black")

    _style_axes(ax)

    leg = ax.legend(frameon=False, loc="lower right")
    for text in leg.get_texts():
        text.set_color("black")

    ax.set_xlabel("False Positive Rate", fontsize=12, color="black")
    ax.set_ylabel("True Positive Rate", fontsize=12, color="black")
    ax.set_title(title, fontsize=13, color="black")

    _style_axes(ax)
    ax.legend(frameon=False, loc="lower right")
    fig.tight_layout()
    _save_pub(fig, out_prefix)
    plt.close(fig)



# --------------------------- Plot PR -----------------------------------------

def plot_cv_pr(res_pr, out_prefix, title="Scaffold-grouped 10-fold CV Precision–Recall"):
    fig, ax = plt.subplots(figsize=(7.2, 6.2))

    # Fold curves (grey)
    for rec, prec, ap in zip(res_pr["fold_rec"], res_pr["fold_prec"], res_pr["fold_ap"]):
        ax.plot(rec, prec, linewidth=1, alpha=0.25, color="grey")

    # Mean curve (blue)
    mrec = res_pr["mean_rec"]
    mprec = res_pr["mean_prec"]
    sprec = res_pr["std_prec"]

    ax.plot(
        mrec, mprec,
        linewidth=2.5,
        color="blue",
        label=f"Mean PR (AP={res_pr['mean_ap']:.3f}±{res_pr['std_ap']:.3f})"
    )

    ax.fill_between(
        mrec,
        np.clip(mprec - sprec, 0, 1),
        np.clip(mprec + sprec, 0, 1),
        color="lightblue",
        alpha=0.30,
        linewidth=0
    )

    # Baseline precision (RED dashed)
    base = res_pr["baseline_prec"]
    ax.hlines(
        base, 0, 1,
        linestyles="--",
        linewidth=1.5,
        color="red",
        label=f"Baseline precision (pos frac={base:.3f})"
    )

    ax.set_xlabel("Recall", fontsize=12, color="black")
    ax.set_ylabel("Precision", fontsize=12, color="black")
    ax.set_title(title, fontsize=13, color="black")

    _style_axes(ax)

    leg = ax.legend(frameon=False, loc="lower left")
    for text in leg.get_texts():
        text.set_color("black")

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)

    fig.tight_layout()
    _save_pub(fig, out_prefix)
    plt.close(fig)

plot_cv_roc(res["roc"], OUT_ROC_PREFIX)
plot_cv_pr(res["pr"], OUT_PR_PREFIX)

[LOAD] C:\Users\Besitzer\Desktop\M3_databases\ChEMBL_M3_consensus_labels_more_negatives_with_meta.csv | shape = (2268, 13)
[RDKit] building mols + scaffolds ...
[RDKit] building fingerprints ...
[DATA] n = 2268 | pos = 1788 | neg = 480 | pos_frac = 0.7883597883597884
[SCAFF] unique scaffolds = 1131


[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerator
[13:19:52] DEPRECATION WARNING: please use MorganGenerat

[Fold 01] ROC-AUC=0.946 | AP(PR-AUC)=0.988 | n_test=227 | pos_test=205
[Fold 02] ROC-AUC=0.991 | AP(PR-AUC)=0.997 | n_test=227 | pos_test=177
[Fold 03] ROC-AUC=0.984 | AP(PR-AUC)=0.998 | n_test=227 | pos_test=197
[Fold 04] ROC-AUC=0.884 | AP(PR-AUC)=0.955 | n_test=227 | pos_test=171
[Fold 05] ROC-AUC=0.956 | AP(PR-AUC)=0.992 | n_test=227 | pos_test=195
[Fold 06] ROC-AUC=0.974 | AP(PR-AUC)=0.991 | n_test=227 | pos_test=173
[Fold 07] ROC-AUC=0.970 | AP(PR-AUC)=0.988 | n_test=227 | pos_test=158
[Fold 08] ROC-AUC=0.972 | AP(PR-AUC)=0.984 | n_test=227 | pos_test=152
[Fold 09] ROC-AUC=0.982 | AP(PR-AUC)=0.996 | n_test=226 | pos_test=187
[Fold 10] ROC-AUC=0.961 | AP(PR-AUC)=0.967 | n_test=226 | pos_test=173
[SAVED] C:\Users\Besitzer\Desktop\M3_databases\figures\CV_curves\CV10_scaffold_ROC.png
[SAVED] C:\Users\Besitzer\Desktop\M3_databases\figures\CV_curves\CV10_scaffold_ROC.pdf
[SAVED] C:\Users\Besitzer\Desktop\M3_databases\figures\CV_curves\CV10_scaffold_PR.png
[SAVED] C:\Users\Besitzer\Desk